https://docs.langchain.com/oss/python/langchain/tools#toolnode

#### ToolNode
- tool execution, error handling, state injection

In [ ]:
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))

# Create the ToolNode with your tools
tool_node = ToolNode([search, calculator])

# Use in a graph
builder = StateGraph(MessagesState)
builder.add_node("tools", tool_node)
# ... add other nodes and edges

#### tool의 3가지 return값
1. string
- ToolMessage로 바뀜
- state를 바꾸지 않음

2. object ex. dict
- tool output에 str형태로 나옴
- 모델은 field명을 인식하고, 이를 바탕으로 추론가능
- state를 바꾸지 않음

3. Command
- state를 update해야할 때 Command를 return
- ToolMessage를 포함/미포함하여 Command를 return할 수 있음
- ToolMessage 포함예시: 하단참고
- state를 바꿈
- 같은 run에서 그 다음 모든 state에 update된 state가 적용됨
- parallel tool calls에 의해 State가 update되는 것을 방지하려면 reducer를 사용할 것  ex) add_messages, operator.add  
  
참고: https://docs.langchain.com/oss/python/langgraph/graph-api#reducers

In [1]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def set_language(language: str, runtime: ToolRuntime) -> Command:
    """Set the preferred response language."""
    return Command(
        update={
            "preferred_language": language,
            "messages": [
                ToolMessage(
                    content=f"Language set to {language}.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

#### Error handling
- 공식문서 내 예시를 참고할 것

#### Route with tools_condition
- tools_condition 함수를 conditional edge에 쓰면 tool 또는 END 노드로 라우팅 함
- TODO: 어떤 원리로 하는 건 지는 공식문서 찾아봐야 함

#### State injection; ToolRuntime
- ToolRuntime을 통해 tool에서 현재 state값에 접근가능함

In [3]:
# 신기하다

from langchain.tools import tool, ToolRuntime
from langgraph.prebuilt import ToolNode

@tool
def get_message_count(runtime: ToolRuntime) -> str:
    """Get the number of messages in the conversation."""
    messages = runtime.state["messages"]
    return f"There are {len(messages)} messages."

tool_node = ToolNode([get_message_count])